# CNMFe pipeline — part 1: load & motion-correct

This is the **starting point** for new users. It walks through the first half
of the CNMFe pipeline on a single recording session:

1. **Ingest** the recording into a zarr store. Two options:
   - **AVI** — point at a folder of numbered AVIs (`0.avi`, `1.avi`, ...). The
     concatenator streams them into one zarr (canonical pipeline format).
   - **Zarr** — already converted? Just open it.
2. **Inspect** the zarr — shape/chunks/dtype, sample frames, mean projection.
3. **Motion-correct** the movie via streaming (RAM-bounded) `CNMFe.fit_mc(...)`.
4. **Verify** correction quality — shifts over time, peak-shift frame comparison,
   mean-projection sharpness before vs after.

The corrected zarr written by this notebook is the input for the next part of
the pipeline (initialization → ring background → spatial/temporal updates).

## 1. Imports

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np

from minicnmfe.io import open_zarr
from minicnmfe.pipeline import CNMFe, CNMFeParams

## 2. Choose your input

Set `INPUT_MODE` to one of:

- `'avi'` — concatenate a folder of numbered AVIs (`0.avi`, `1.avi`, ...) into a
  single time-chunked zarr store. Idempotent: skips re-encoding if the output
  zarr already exists.
- `'zarr'` — load an existing zarr store directly. Use this when you've already
  run the AVI→zarr step (in this notebook or via `concat_avis_to_zarr.py`).

The default below points at the bundled `demo_movies/demo_session/` folder so
this notebook runs out-of-the-box. Replace with your own session paths.

In [ ]:
# --- Pick one ----------------------------------------------------------------
INPUT_MODE = 'avi'                  # 'avi'  or  'zarr'

PROJECT_ROOT = Path('D:/code/claude_cnmfe')

# Path used when INPUT_MODE == 'avi'.
# Folder containing numerically-named AVI files (0.avi, 1.avi, ...).
AVI_FOLDER  = PROJECT_ROOT / 'demo_movies' / 'demo_session'
AVI_PATTERN = '*.avi'

# Path used when INPUT_MODE == 'zarr'.
# Direct path to an existing zarr store.
ZARR_PATH   = PROJECT_ROOT / 'demo_movies' / 'demo_session' / 'session.zarr'

# --- Output (used by both branches) -----------------------------------------
# Where the concatenated zarr ('avi' branch) and the motion-corrected zarr live.
if INPUT_MODE == 'avi':
    CONCAT_ZARR = AVI_FOLDER / 'session.zarr'
    OUTPUT_DIR  = AVI_FOLDER / 'mc_output'
elif INPUT_MODE == 'zarr':
    CONCAT_ZARR = ZARR_PATH
    OUTPUT_DIR  = ZARR_PATH.parent / 'mc_output'
else:
    raise ValueError(f"INPUT_MODE must be 'avi' or 'zarr', got {INPUT_MODE!r}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'INPUT_MODE  : {INPUT_MODE}')
print(f'source zarr : {CONCAT_ZARR}')
print(f'output dir  : {OUTPUT_DIR}')

## 3. Ingest — AVI → zarr (only runs when `INPUT_MODE == 'avi'`)

Two steps:

**3a.** Discover and list the AVI files so you can sanity-check counts and
spatial dimensions before kicking off the (slower) concat step.

**3b.** Stream all AVIs into one zarr store. Each file is decoded frame-by-frame
and written to a time-chunked, blosc-compressed zarr at `CONCAT_ZARR`. Peak RAM
stays bounded by `chunk_t` regardless of how many frames you have.
`skip_if_exists=True` makes this cell idempotent — rerun freely.

Skip this whole section when `INPUT_MODE == 'zarr'`.

In [ ]:
if INPUT_MODE == 'avi':
    from concat_avis_to_zarr import _count_and_shape, _numeric_key

    avis = sorted(AVI_FOLDER.glob(AVI_PATTERN), key=_numeric_key)
    avis = [p for p in avis if _numeric_key(p) >= 0]
    assert avis, f'No numerically-named AVIs in {AVI_FOLDER}'

    print(f'{len(avis)} AVI file(s) found in {AVI_FOLDER}:\n')
    total = 0
    H = W = None
    for p in avis:
        n, H, W = _count_and_shape(p)
        total += n
        print(f'  {p.name:>20s}   {n:>6d} frames   {H}x{W}')
    print(f'\n  TOTAL              {total:>6d} frames   {H}x{W}')
else:
    print(f"INPUT_MODE == {INPUT_MODE!r}; skipping AVI discovery.")

In [ ]:
if INPUT_MODE == 'avi':
    from concat_avis_to_zarr import concat_avis_to_zarr

    t0 = time.time()
    concat_avis_to_zarr(
        folder=AVI_FOLDER,
        output_path=CONCAT_ZARR,
        pattern=AVI_PATTERN,
        chunk_t=200,           # tune for IO/RAM balance; 200 is a good default
        dtype='uint8',         # keep 8-bit on disk; float32 conversion happens in RAM
        grayscale=True,
        skip_if_exists=True,   # rerun-safe: no re-encode if CONCAT_ZARR already exists
        verbose=True,
    )
    print(f'\nelapsed: {time.time() - t0:.1f}s')
else:
    print(f"INPUT_MODE == {INPUT_MODE!r}; skipping AVI concat.")

## 4. Inspect the zarr

From here on, both branches converge: `CONCAT_ZARR` points at the canonical
input store. We open it lazily (no decode until we actually index into it),
print its layout, and visualize a few sample frames + the mean projection.

A few things to look for:

- **shape `(T, H, W)`** — total frames and spatial dimensions match expectations.
- **`chunks`** — time chunk size. Bigger = fewer reads but more RAM per read.
- **dtype** — usually `uint8` for raw miniscope; pipeline upcasts to float32 in RAM.
- **sample frames** — flicker/noise pattern looks like 1p calcium imaging.
- **mean projection** — neurons + vasculature visible; obvious drift shows up
  as smeared bright structures.

In [ ]:
z = open_zarr(CONCAT_ZARR)
T, H, W = z.shape
print(f'shape  : {z.shape}')
print(f'chunks : {z.chunks}')
print(f'dtype  : {z.dtype}')
print(f'on-disk size : ~{z.nbytes / 1e6:.0f} MB (uncompressed equivalent)')

In [ ]:
# Four evenly-spaced sample frames (only these slices are decoded).
sample_idx = np.linspace(0, T - 1, 4).astype(int)
sample = np.stack([np.asarray(z[t]) for t in sample_idx], axis=0)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, t, img in zip(axes, sample_idx, sample):
    ax.imshow(img, cmap='gray')
    ax.set_title(f't = {t}')
    ax.axis('off')
fig.suptitle('Sample frames from the session zarr', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Mean projection — streamed in batches so RAM stays bounded.
def mean_projection(zarr_arr, batch_size=500):
    T = zarr_arr.shape[0]
    acc = np.zeros(zarr_arr.shape[1:], dtype=np.float64)
    for start in range(0, T, batch_size):
        end = min(start + batch_size, T)
        acc += np.asarray(zarr_arr[start:end], dtype=np.float64).sum(axis=0)
    return (acc / T).astype(np.float32)

raw_mean = mean_projection(z)

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.imshow(raw_mean, cmap='gray')
ax.set_title('Mean projection (raw, pre-MC)')
ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Configure motion correction

Motion correction (MC) estimates a rigid `(dy, dx)` shift per frame against a
median template and warps each frame back. Because the input is a `zarr.Array`,
`CNMFe.fit_mc` automatically takes the **streaming** path: frames are read,
corrected, and written to a new zarr in batches; the full movie never lives in
RAM.

Key knobs:

- **`max_shift`** — clamp maximum allowed `(dy, dx)` per frame. Make it a bit
  bigger than the worst drift you expect.
- **`mc_gSig_filt`** — sigma of the spatial high-pass applied before cross-
  correlation. **Required for 1p** data (suppresses slow background). Set to
  `None` for 2p. A reasonable default is ≈ neuron radius in pixels.
- **`mc_batch_size`** — frames per streaming/parallel batch. Bigger ≈ faster
  but more RAM.
- **`mc_template_max_frames`** — cap on frames sampled (strided) for the median
  template. Bounds template-step RAM independent of `T`.
- **`n_jobs`** — CPU workers for per-frame work. `-1` = all cores.
- **`upsample_factor`** — subpixel refinement (10 ≈ 0.1 px precision).

Peak RAM ≈ `(mc_batch_size + mc_template_max_frames) * H * W * 4` bytes.

In [ ]:
params = CNMFeParams(
    max_shift=(20, 20),
    upsample_factor=10,
    mc_n_iter=1,                     # 1 pass is the CaImAn default
    mc_gSig_filt=7,                  # 1p high-pass sigma; set to None for 2p
    mc_batch_size=200,
    mc_template_max_frames=2000,
    mc_output_chunk_t=None,          # None = match source chunks
    mc_output_dtype='float32',
    n_jobs=-1,
)

peak_mb = (params.mc_batch_size + params.mc_template_max_frames) * H * W * 4 / 1e6
print(f'estimated peak RAM (MC step): ~{peak_mb:.0f} MB')

## 6. Run streaming motion correction

`CNMFe.fit_mc(zarr, output_dir=...)` reads the source zarr in batches,
estimates per-frame `(dy, dx)` shifts against a strided-median template, warps
each frame, and writes the corrected frames to `<output_dir>/mc.zarr`.

Returned: a `zarr.Array` handle to `mc.zarr`. Shifts live on `model.shifts`
as a `(T, 2)` float32 array.

In [ ]:
model = CNMFe(params)

t0 = time.time()
mc = model.fit_mc(z, output_dir=OUTPUT_DIR)
mc_elapsed = time.time() - t0

print(f'\nMC finished in {mc_elapsed:.1f}s')
print(f'  corrected zarr : {OUTPUT_DIR / "mc.zarr"}')
print(f'  mc.shape       : {mc.shape}')
print(f'  mc.chunks      : {mc.chunks}')
print(f'  mc.dtype       : {mc.dtype}')
print(f'  model.shifts   : {model.shifts.shape}  {model.shifts.dtype}')

## 7. Verify MC quality

Three quick checks:

- **Shift trajectory** — `(dy, dx)` per frame. Slow drift + small jitter is
  expected; sudden jumps near `max_shift` suggest you should raise the cap.
- **Peak-shift frame** — the frame with the largest correction, raw vs corrected.
  Neurons should look stable (no smearing, no ringing) in the corrected version.
- **Mean projection before vs after** — corrected projection should be visibly
  *sharper* (small bright structures less smeared).

In [ ]:
shifts = model.shifts   # (T, 2) — (dy, dx) per frame

fig, ax = plt.subplots(1, 1, figsize=(10, 3.5))
ax.plot(shifts[:, 0], label='dy', lw=1)
ax.plot(shifts[:, 1], label='dx', lw=1)
ax.set_xlabel('frame')
ax.set_ylabel('shift (px)')
ax.set_title('Per-frame correction shifts')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'  dy : min={shifts[:, 0].min():+.2f}  max={shifts[:, 0].max():+.2f}'
      f'   std={shifts[:, 0].std():.2f}')
print(f'  dx : min={shifts[:, 1].min():+.2f}  max={shifts[:, 1].max():+.2f}'
      f'   std={shifts[:, 1].std():.2f}')

In [ ]:
peak_t = int(np.argmax(np.linalg.norm(shifts, axis=1)))
raw_peak = np.asarray(z[peak_t], dtype=np.float32)
mc_peak  = np.asarray(mc[peak_t], dtype=np.float32)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(raw_peak, cmap='gray')
axes[0].set_title(f'raw   (t={peak_t}, shift={shifts[peak_t]})')
axes[0].axis('off')
axes[1].imshow(mc_peak, cmap='gray')
axes[1].set_title('corrected')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
mc_mean = mean_projection(mc)

vmin = min(raw_mean.min(), mc_mean.min())
vmax = max(raw_mean.max(), mc_mean.max())

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(raw_mean, cmap='gray', vmin=vmin, vmax=vmax)
axes[0].set_title('mean projection (raw)')
axes[0].axis('off')
axes[1].imshow(mc_mean, cmap='gray', vmin=vmin, vmax=vmax)
axes[1].set_title('mean projection (corrected)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 8. Save shifts to disk

Persist `model.shifts` next to `mc.zarr` so the next notebook (or external
analysis that correlates behaviour with frame motion) can pick them up without
rerunning MC.

In [ ]:
shifts_path = OUTPUT_DIR / 'shifts.npy'
np.save(shifts_path, model.shifts)
print(f'saved -> {shifts_path}')
print(f'         shape={model.shifts.shape}  dtype={model.shifts.dtype}')

## 9. What's next

- The corrected movie lives at `<OUTPUT_DIR>/mc.zarr` as a normal zarr store.
  Reload it later with `open_zarr(...)`.
- Per-frame shifts are at `<OUTPUT_DIR>/shifts.npy`.
- The follow-up notebook (`02_extract_components.ipynb`) picks up here and
  runs preprocessing (CORR/PNR), greedy initialization, ring-background
  subtraction, and the spatial / temporal updates. Pass
  `do_motion_correction=False` to `CNMFe.fit(...)` when feeding it the
  already-corrected zarr.
- **For very long recordings (60k+ frames)**, run
  `minicnmfe.io.transpose_zarr_to_pixel_major(<OUTPUT_DIR>/'mc.zarr', <OUTPUT_DIR>/'mc_pixel.zarr')`
  once after this notebook. Part 2 can then call
  `fit(mc_zarr, do_motion_correction=False, Y_flat_zarr=mc_pixel_zarr)` to
  run extraction without ever materialising the full `(T, H, W)` movie
  in RAM. See part 2's Section 9.